# Olist Logistics Agent

Foco exclusivo na análise de prazos de entrega, atrasos e performance logística para entender gargalos da cadeia de distribuição.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')

## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes de logística

In [ ]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('orders', orders.shape)
print('order_items', order_items.shape)
print('sellers', sellers.shape)

## Preparar métricas de entrega

Converter datas e calcular tempos de entrega para análise logística.

In [ ]:
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

orders['time_to_carrier'] = (orders['order_delivered_carrier_date'] - orders['order_approved_at']).dt.days
orders['time_to_customer'] = (orders['order_delivered_customer_date'] - orders['order_approved_at']).dt.days
orders['carrier_to_customer'] = (orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']).dt.days

orders[['time_to_carrier', 'time_to_customer', 'carrier_to_customer']].describe()

# Tempo por etapa logística do seller para o cliente

Neste bloco, entramos na análise de pedidos atrasados para entender como o tempo é distribuído entre o seller e o parceiro logístico. O código a seguir:
- junta `order_items` com `sellers` para trazer o `shipping_limit_date` por pedido e identificar o seller associado;
- remove duplicatas de `order_id` + `seller_id`, mantendo um registro único por pedido e seller;
- une essa informação com a tabela `orders` para trabalhar com as datas de aprovação, envio e entrega.

Em `delayed_orders`, filtramos apenas os pedidos com todas as datas necessárias preenchidas e que entregaram após a data estimada (`delivery_delay_vs_estimate > 0`). Depois, calculamos:
- `seller_handling_days` = dias entre aprovação do pedido e repasse ao transportador;
- `logistics_days` = dias entre entrega ao transportador e entrega ao cliente;
- `seller_deadline_gap` = diferença entre `order_delivered_carrier_date` e `shipping_limit_date`, para classificar o seller como on time ou late;
- `delivery_delay_vs_estimate` = atraso final em relação à data estimada ao cliente;
- `seller_pct` e `logistics_pct` = participação de cada etapa no tempo total do pedido.

Em `status_summary`, agregamos por `seller_status` para comparar:
- volume de pedidos atrasados;
- atraso médio final;
- tempo médio do seller e do parceiro logístico;
- participação média de cada etapa no tempo total.

Esse bloco entrega uma visão clara de quanto do atraso final decorre de atraso do seller versus atraso da etapa logística, usando apenas pedidos completos e atrasados em relação à previsão do cliente.

In [ ]:
order_sellers = pd.merge(order_items[['order_id', 'seller_id', 'shipping_limit_date']], sellers[['seller_id', 'seller_label']], on='seller_id', how='left')
order_sellers = order_sellers.drop_duplicates(subset=['order_id', 'seller_id'])
order_with_sellers = pd.merge(orders, order_sellers, on='order_id', how='left')

# Filtrar apenas os pedidos que atrasaram em relação ao prazo estimado
delayed_orders = (
    order_with_sellers.dropna(subset=[
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
        'shipping_limit_date'
    ])
    .assign(
        seller_handling_days=lambda df: (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.days,
        logistics_days=lambda df: (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.days,
        actual_total_days=lambda df: df['seller_handling_days'] + df['logistics_days'],
        estimated_total_days=lambda df: (df['order_estimated_delivery_date'] - df['order_approved_at']).dt.days,
        seller_deadline_gap=lambda df: (df['order_delivered_carrier_date'] - df['shipping_limit_date']).dt.days,
        delivery_delay_vs_estimate=lambda df: (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days,
        seller_pct=lambda df: np.where(df['actual_total_days'] > 0, df['seller_handling_days'] / df['actual_total_days'], 0),
        logistics_pct=lambda df: np.where(df['actual_total_days'] > 0, df['logistics_days'] / df['actual_total_days'], 0),
        seller_status=lambda df: np.where(df['seller_deadline_gap'] <= 0, 'Seller on time', 'Seller late')
    )
    .query('delivery_delay_vs_estimate > 0')
)

status_summary = (
    delayed_orders.groupby('seller_status')
    .agg(
        delayed_order_count=('order_id', 'nunique'),
        avg_delay_days=('delivery_delay_vs_estimate', 'mean'),
        avg_seller_days=('seller_handling_days', 'mean'),
        avg_logistics_days=('logistics_days', 'mean'),
        avg_seller_pct=('seller_pct', 'mean'),
        avg_logistics_pct=('logistics_pct', 'mean')
    )
    .reset_index()
)

status_summary = status_summary.assign(
    pct_delayed=lambda df: df['delayed_order_count'] / df['delayed_order_count'].sum() * 100
)

status_summary

### 1) Volume de pedidos atrasados com o Sellers

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=status_summary, x='seller_status', y='delayed_order_count', palette=['#4c72b0', '#d62728'])
plt.title('Volume de pedidos atrasados além do prazo estimado por status do seller')
plt.xlabel('Status do seller')
plt.ylabel('Pedidos atrasados')
percentages = status_summary['delayed_order_count'] / status_summary['delayed_order_count'].sum() * 100
for p, pct in zip(ax.patches, percentages):
    ax.annotate(
        f'{pct:.1f}%',
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center',
        va='bottom',
        fontsize=11,
        color='black',
        xytext=(0, 5),
        textcoords='offset points'
    )
plt.tight_layout()
plt.show()

### 2) Atraso médio comparados com o tempo médio do Seller e Logística

In [ ]:
metric_data = status_summary.melt(
    id_vars='seller_status',
    value_vars=['avg_delay_days', 'avg_seller_days', 'avg_logistics_days'],
    var_name='metric',
    value_name='days'
)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=metric_data, x='seller_status', y='days', hue='metric', palette=['#7f7f7f', '#4c72b0', '#d62728'])
ax.set_title('Atraso médio e tempo médio de seller/logística para pedidos atrasados')
ax.set_xlabel('Status do seller')
ax.set_ylabel('Dias')
plt.legend(title='Métrica', loc='upper right')
plt.tight_layout()
plt.show()

### Rotas críticas por estado

Agora que a análise anterior mostra que a maior parte dos atrasos vem da etapa logística, vamos identificar quais rotas entre `seller_state → customer_state` estão acumulando mais problemas.

O novo gráfico foca apenas em pedidos que entregaram depois da data estimada (`delivery_delay_vs_estimate > 0`) e calcula um score de rotas problemáticas que combina:
- tempo médio logístico (`avg_logistics_days`);
- número de pedidos atrasados na rota (`delayed_order_count`);
- atraso médio final em relação à estimativa (`avg_delay_vs_estimate`).

O objetivo é destacar rotas que têm não só tempo logístico elevado, mas também volume significativo de pedidos atrasados — rotas com maior potencial de intervenção operacional.


In [ ]:
customers = pd.read_csv(base_path + 'olist_customers_dataset.csv')[['customer_id', 'customer_state']]
order_sellers = pd.merge(
    order_items[['order_id', 'seller_id', 'shipping_limit_date']],
    sellers[['seller_id', 'seller_state']],
    on='seller_id',
    how='left'
)
order_sellers = order_sellers.drop_duplicates(subset=['order_id', 'seller_id'])
order_with_sellers = pd.merge(orders, order_sellers, on='order_id', how='left')
order_with_customer = pd.merge(order_with_sellers, customers, on='customer_id', how='left')

route_problem_metrics = (
    order_with_customer.dropna(subset=['order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'seller_state', 'customer_state'])
    .assign(
        logistics_days=lambda df: (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.days,
        delivery_delay_vs_estimate=lambda df: (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
    )
    .query('delivery_delay_vs_estimate > 0')
    .groupby(['seller_state', 'customer_state'])
    .agg(
        delayed_order_count=('order_id', 'nunique'),
        avg_logistics_days=('logistics_days', 'mean'),
        avg_delay_vs_estimate=('delivery_delay_vs_estimate', 'mean')
    )
    .reset_index()
)

route_problem_metrics = route_problem_metrics.assign(
    route_problem_score=lambda df: df['avg_logistics_days'] * df['delayed_order_count']
)

problem_routes = route_problem_metrics.sort_values('route_problem_score', ascending=False).head(20)
problem_routes['route_label'] = problem_routes['seller_state'] + ' → ' + problem_routes['customer_state']

plt.figure(figsize=(14, 10))
sns.scatterplot(
    data=problem_routes,
    x='avg_logistics_days',
    y='route_label',
    size='delayed_order_count',
    hue='avg_delay_vs_estimate',
    palette='coolwarm',
    sizes=(100, 800),
    legend='brief'
)
plt.title('Top 20 rotas problemáticas: logística média x pedidos atrasados')
plt.xlabel('Dias médios de logística')
plt.ylabel('Rota (seller_state → customer_state)')
plt.legend(title='Atraso médio vs estimativa', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Esse gráfico destaca rotas que têm um comportamento mais crítico em entregas atrasadas. A combinação de alto tempo médio logístico e volume de pedidos atrasados ajuda a priorizar rotas que devem ser investigadas primeiro.